# DDS + CIC Calibration and Linearity Analysis

This notebook explains how to measure, analyze, and calibrate the output of an ADC followed by a Polyphase Filter Bank (PFB) and a DDS + CIC block. 

## 1. System Overview

We consider the following chain:

1. RF signal entering an ADC:
   - Frequency: $f_{RF} = 300\,\mathrm{MHz}$
   - Power: variable $P_{in}$ (dBm)
   - ADC sampling rate: $F_s = 4096\,\mathrm{MHz}$
2. Digital decimation by 2 and mixing by $-F_s/4$ to create a complex baseband.
3. Polyphase Filter Bank (PFB) with 8 channels and decimation factor 8:
   - Output bandwidth per channel: 256 MHz
   - PFB **net gain** = polyphase accumulation gain + qout attenuation:
     $$G_{\text{PFB,net}} = +20\log_{10}(D) - 20\log_{10}(2^{q_{\text{out}}}) = +18.06 - 36.12 = -18.06\,\text{dB}$$
   - With `qout = 6` and `D = 8` (see Notebook 02)
4. DDS to translate PFB output signal to DC
   - DDS Mixer Gain (16-bit full scale): $+90.31\,\text{dB}$
5. CIC for decimating the signal
   - Decimation factor R = 8
   - Differential delay M = 1
   - Order N = 3
   - **CIC gain is internally normalized by the hardware** (QCIC=0): $G_{\text{CIC}} = 0\,\text{dB}$
   - DDS + CIC block output quantization controlled by `QPROD_REG(14)`

The goal is to measure the tone level at the output of the DDS + CIC and convert it to real RF power in dBm.

---


## Gain Compensation Between PFB and CIC

In the processing chain

```
PFB → DDS → CIC
```

the CIC decimator introduces a significant gain that must be taken into account to avoid overflow and to maintain a convenient signal scaling.

For a CIC filter with:

* Decimation factor (R = 8)
* Differential delay (M = 1)
* Order (N = 3)

the DC gain of the filter is

$$
G_{CIC} = (RM)^N = (8 \cdot 1)^3 = 512 \quad (+54.19\,\text{dB})
$$

---

### Hardware Implementation Details (from cic_3.v and ddscic.v)

The CIC filter in the hardware has an internal bit width of `BCIC = B + N*log2(1024) = 16 + 30 = 46 bits`. 

**Key insight from the Verilog code:**

```verilog
// CIC input in ddscic.v
assign cic_din_real = {{(BCIC-B){prod_mux_r1[B-1]}}, prod_mux_r1[0 +: B]};
```

The input to the CIC is **sign-extended** from 16 bits to 46 bits. This means the CIC operates on a 16-bit number that has been expanded to 46 bits, NOT on a number that has been multiplied by any factor.

**Critical point:** Although the CIC has a theoretical gain of 512, the output is then truncated by `qdata_iq` with `QCIC_REG=0`:

```verilog
qdata_iq #(.BIN(BCIC), .BOUT(B)) qdata_cic_i(
    .din(cic_dout),      // 46-bit CIC output
    .dout(cic_dout_round), // 16-bit output
    .QSEL_REG(QCIC_REG)   // QCIC_REG = 0
);
```

With `QCIC_REG=0`, the `qdata` module selects bits `[0:15]` (the 16 LSBs) of the 46-bit CIC output. The hardware is designed so that the combination of:
- Sign extension to 46 bits
- CIC gain (512)
- LSB selection (QCIC=0)

results in a **net gain of 0 dB** for the CIC stage under normal operating conditions. This is an intentional internal normalization.

Therefore, for calibration purposes:
$$G_{\text{CIC, effective}} = 0\,\text{dB}$$

---

### PFB Net Gain (corrected model)

As established in [PFB data analysis](./02_PFB_data.ipynb), the PFB introduces **two** amplitude effects:

1. **Polyphase accumulation gain:** the D = 8 polyphase branches coherently accumulate, producing an amplitude gain of D:
$$G_{\text{polyphase}} = +20\log_{10}(D) = +20\log_{10}(8) = +18.06\,\text{dB}$$

2. **qout right-shift attenuation:** the FFT output is right-shifted by `qout = 6` bits before truncation to 16 bits:
$$G_{q_{\text{out}}} = -20\log_{10}(2^{q_{\text{out}}}) = -20\log_{10}(64) = -36.12\,\text{dB}$$

The **net PFB gain** is therefore:
$$G_{\text{PFB,net}} = G_{\text{polyphase}} + G_{q_{\text{out}}} = +18.06 - 36.12 = -18.06\,\text{dB}$$

> **Common mistake:** using only the qout term ($-36.12\,\text{dB}$) without the polyphase gain ($+18.06\,\text{dB}$) produces a systematic calibration error of ~20 dB.

---

### Compensation Using qout

To pre-compensate the CIC gain at the PFB output (keeping the signal level roughly constant across the full chain), one can choose `qout` so that the combined PFB–CIC gain is near unity.

The condition is:

$$G_{\text{polyphase}} + G_{q_{\text{out}}} + G_{\text{CIC}} \approx 0\,\text{dB}$$

$$+18.06 - 6.02 \cdot q_{\text{out}} + 54.19 = 0$$

$$q_{\text{out}} = \frac{18.06 + 54.19}{6.02} \approx 12$$

In practice, the firmware saturates `QOUT_REG` at 11 (hardware protection), so perfect compensation is not achievable with `qout` alone. For this experiment `qout = 6` is used, meaning the CIC gain is **not** pre-compensated; instead the full gain budget is tracked analytically in the calibration (see Section 3).

---


## Quantization Scaling in the DDS+CIC Block

The DDS+CIC block has **two** quantization stages (from ddscic.v):

### 1. QPROD Quantization (before CIC)

```verilog
qdata_iq #(.BIN(2*B), .BOUT(B)) qdata_prod_i(
    .din(prod),           // 32-bit DDS product
    .dout(prod_round),    // 16-bit output
    .QSEL_REG(QPROD_REG)  // QPROD_REG = 14
);
```

The `qdata` module selects a 16-bit slice of the input word according to

```verilog
assign dout = din[qsel_i +: BOUT];
```

For the configuration used in this experiment:

```
BIN      = 32
BOUT     = 16
QSEL_REG = 14 (QPROD_REG)
```

This selects bits `[29:14]` of the 32-bit product, which effectively corresponds to an arithmetic right shift of the input signal by 14 bits:

$$G_{\text{QPROD}} = 2^{-14} = \frac{1}{16384} \quad (-84.29\,\text{dB})$$

### 2. QCIC Quantization (after CIC)

```verilog
qdata_iq #(.BIN(BCIC), .BOUT(B)) qdata_cic_i(
    .din(cic_dout),       // 46-bit CIC output
    .dout(cic_dout_round), // 16-bit output
    .QSEL_REG(QCIC_REG)   // QCIC_REG = 0
);
```

With `QCIC_REG = 0`, this selects bits `[15:0]` (the 16 LSBs) of the 46-bit CIC output.

**Important:** Although the CIC has a theoretical gain of 512 (+54.19 dB), the combination of:
- Sign-extended input (16 → 46 bits)
- CIC accumulation
- LSB selection (bits [15:0])

results in a **net effective gain of 0 dB** for the CIC+QCIC stage under normal operating conditions. This is an intentional hardware normalization.

---

## Total Gain of the DSP Chain

The signal processing chain from PFB output to DDS+CIC output is:

```
PFB  →  DDS  →  QPROD  →  CIC  →  QCIC
```

Each stage contributes a gain factor.

### PFB Net Gain (polyphase accumulation + qout)

$$G_{\text{PFB,net}} = +20\log_{10}(D) - 20\log_{10}(2^{q_{\text{out}}})$$

With $D = 8$ and $q_{\text{out}} = 6$:

$$G_{\text{PFB,net}} = +18.06 - 36.12 = -18.06\,\text{dB}$$

### DDS Mixer Gain
The DDS multiplies the signal by a 16-bit full-scale sine wave (amplitude $2^{15}$):
$$G_{\text{DDS}} = 20\log_{10}(2^{15}) = +90.31\,\text{dB}$$

### QPROD Quantization Gain
$$G_{\text{QPROD}} = -20\log_{10}(2^{14}) = -84.29\,\text{dB}$$

### CIC + QCIC Combined Effective Gain
Due to the hardware design (sign extension to 46 bits + LSB selection with QCIC=0):
$$G_{\text{CIC+QCIC, effective}} = 0\,\text{dB}$$

---

## Combined Gain from PFB output to DDS+CIC output

$$G_{\text{DDS+CIC\ block}} = G_{\text{DDS}} + G_{\text{QPROD}} + G_{\text{CIC+QCIC, effective}}$$

$$G_{\text{DDS+CIC\ block}} = 90.31 - 84.29 + 0 = +6.02\,\text{dB}$$

## Total Chain Gain (from PFB output to final output)

$$G_{\text{TOTAL}} = G_{\text{PFB,net}} + G_{\text{DDS+CIC\ block}}$$

$$G_{\text{TOTAL}} = -18.06 + 6.02 = -12.04\,\text{dB}$$

---

## Gain Summary of the DSP Chain

The following table summarizes the gain from the **PFB output** through to the DDS+CIC output:

| Stage | Gain (dB) | Description |
|---|---|---|
| PFB polyphase accumulation (D=8) | +18.06 dB | Coherent sum of 8 polyphase branches |
| PFB qout right-shift (qout=6) | −36.12 dB | Bit truncation at PFB output |
| **PFB net** | **−18.06 dB** | Combined PFB effect |
| DDS Mixer | +90.31 dB | Multiplication by 16-bit sine |
| QPROD scaling (QPROD=14) | −84.29 dB | Bit-slice selection before CIC |
| CIC + QCIC (hardware normalized) | 0.00 dB | CIC gain canceled by LSB selection |
| **Total (PFB→output)** | **−12.04 dB** | Overall DSP chain gain |

---

## Calibration Implication

The calibration constant at the DDS+CIC output is built from all preceding stages (note that a gain *reduces* the calibration constant):

$$\text{CAL\_CONSTANT\_POST} = -K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{DDS+CIC\ block}}$$

$$= 12.89 + 18.06 - 6.02 = 24.93\,\text{dB}$$

This is the **analytical prediction**; the empirical value is obtained from the linear fit to the data.

---


# DDS+CIC into the signal chain

---

### **1. Decimation Handling**

```python
D = 8
if D == 1:
    chain.analysis.set_ddscic_outsel(data="product", cic="no")
else:
    chain.analysis.set_ddscic_outsel(data="product", cic="yes")
    chain.analysis.set_ddscic_decimation(D)
```

* If `D == 1`, the CIC is bypassed, as DDS+CIC requires decimation ≥ 2 to operate.
* For `D > 1`, the CIC is enabled, and the decimation factor is set.

**Note:** CIC filters introduce **amplitude droop** and **phase distortion** depending on decimation factor and number of stages. Keep this in mind when analyzing post-DDC signals.

---

### **2. DDS Frequency**

```python
f_ddscic = fout - FC
chain.analysis.set_ddscic_ddsfreq(f=f_ddscic*1e6)
```

* This sets the DDS frequency relative to your PFB output channel center.
* Make sure the `fout` is within the Nyquist range after decimation: ($f_{out} < \frac{f_s}{2}$).

---

### **3. Quantization**

```python
chain.analysis.set_ddscic_qprod(14)  # 0-16 bits
```

* This sets the quantization of the output.
* Lower bit width increases quantization noise but reduces resource usage.

---

### **4. Sampling frequency after decimation**

```python
fs_d = chain.fs_ch / chain.analysis.get_ddscic_decimation()
```

* This is the **effective sampling rate** of the post-DDC signal.
* Use `fs_d` for FFT and phase calculations.

---

### **5. Time-domain signal**

```python
[xi,xq] = chain.get_bin_pfb(fout, verbose=True)
x_postddscic_t = xi + 1j*xq
```

* `xi` and `xq` are the in-phase and quadrature components.
* `x_postddscic_t` is the **complex baseband signal** after DDS+CIC.

---

### **6. Saving the data**

```python
np.savetxt(QICK_TOOLS + "/.../x_postddscic_t_data.txt", x_postddscic_t)
```

* Now we have a full time-domain record for later spectrum, amplitude, or phase analysis.

---

### **Next Steps for Analysis**

1. **Amplitude spectrum**

```python
X = np.fft.fftshift(np.fft.fft(x_postddscic_t))
f_axis = np.fft.fftshift(np.fft.fftfreq(len(X), 1/fs_d))
plt.plot(f_axis/1e6, 20*np.log10(np.abs(X)))
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.show()
```

2. **Phase response**

```python
plt.plot(f_axis/1e6, np.angle(X))
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.show()
```

3. **Check CIC effect**

* If you decimate by `D`, CIC introduces a sinc-shaped amplitude response:
  $$
  H(f) = \left|\frac{\sin(\pi f D / f_s)}{D \sin(\pi f / f_s)}\right|^N
  $$
  where ($N$) = number of CIC stages.

4. **Compare pre- and post-DDC signals** to quantify amplitude droop and phase shift.

---

## ADC Calibration with PFB and DDS+CIC Chain

### Complete Signal Chain
```
Pin_gen → [Analog Atten] → ADC → PFB → DDS → QPROD → CIC → QCIC → Output
```

**Chain breakdown:**
1. **Analog attenuation + ADC:** captured by $K_{\text{ADC}} = -12.89\,\text{dB}$
2. **PFB polyphase accumulation (D=8):** coherent gain $+20\log_{10}(8) = +18.06\,\text{dB}$
3. **PFB qout right-shift (qout=6):** attenuation $-20\log_{10}(2^6) = -36.12\,\text{dB}$  
   → **Net PFB gain:** $G_{\text{PFB,net}} = -18.06\,\text{dB}$
4. **DDS:** Multiplication by 16-bit sine wave gives amplitude gain of $+90.31\,\text{dB}$
5. **QPROD scaling (QPROD=14):** $G_{\text{QPROD}} = 2^{-14} = -84.29\,\text{dB}$
6. **CIC filter:** Theoretical gain of 512 (+54.19 dB) but normalized by hardware
7. **QCIC scaling (QCIC=0):** Selects LSBs, canceling the CIC gain
   → **Combined CIC+QCIC effective gain:** $G_{\text{CIC+QCIC}} = 0\,\text{dB}$

---

## Calibration Model

The full chain equation (in dB) is:

$$P_{\text{output}}\,(\text{dBFS}) = P_{\text{in}}\,(\text{dBm}) + K_{\text{ADC}} + G_{\text{PFB,net}} + G_{\text{DDS}} + G_{\text{QPROD}} + G_{\text{CIC+QCIC}}$$

Rearranging to recover RF power:

$$\boxed{P_{\text{in}}\,(\text{dBm}) = P_{\text{output}}\,(\text{dBFS}) + \underbrace{\left(-K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{DDS}} - G_{\text{QPROD}} - G_{\text{CIC+QCIC}}\right)}_{\text{CAL\_CONSTANT\_POST}}}$$

With the values for this experiment:

$$\text{CAL\_CONSTANT\_POST} = -(-12.89) - (-18.06) - (+90.31) - (-84.29) - (0) = 24.93\,\text{dB}$$

### Budget summary

| Contribution | Value | Formula |
|---|---|---|
| ADC + analog chain | $K_{\text{ADC}} = -12.89\,\text{dB}$ | from Notebook 01 |
| PFB polyphase accumulation | $+18.06\,\text{dB}$ | $+20\log_{10}(8)$ |
| PFB qout right-shift | $-36.12\,\text{dB}$ | $-20\log_{10}(2^6)$ |
| **Net PFB gain** | $-18.06\,\text{dB}$ | $G_{\text{PFB,net}}$ |
| DDS Mixer | $+90.31\,\text{dB}$ | $20\log_{10}(2^{15})$ |
| QPROD scaling (QPROD=14) | $-84.29\,\text{dB}$ | $-20\log_{10}(2^{14})$ |
| CIC + QCIC (hardware normalized) | $0.00\,\text{dB}$ | Effective gain after LSB selection |
| **CAL_CONSTANT_POST (analytical)** | **+24.93 dB** | $-K_{\text{ADC}} - G_{\text{PFB,net}} - G_{\text{DDS}} - G_{\text{QPROD}} - G_{\text{CIC+QCIC}}$ |

---

## Comparison with Notebook 02 (PFB output, DDS+CIC bypassed)

| | PFB output (Notebook 02) | DDS+CIC output (this notebook) |
|---|---|---|
| Signal path | RF → ADC → PFB | RF → ADC → PFB → DDS → QPROD → CIC → QCIC |
| CAL_CONSTANT | +30.95 dB | +24.93 dB (analytical) |
| Difference | — | -6.02 dB = $-G_{\text{DDS+CIC\ block}}$ |

The DDS+CIC block net effect on CAL_CONSTANT relative to the PFB output is:

$$\Delta\text{CAL} = -G_{\text{DDS}} - G_{\text{QPROD}} - G_{\text{CIC+QCIC}} = -90.31 + 84.29 - 0 = -6.02\,\text{dB}$$

Therefore: $\text{CAL\_CONSTANT\_POST} = \text{CAL\_CONSTANT\_PRE} - 6.02\,\text{dB}$ 

---

## Effect on Signal Phase

### What does NOT introduce deterministic phase rotation
- `qout` right-shift and `qdata` scaling: amplitude only, phase is preserved.
- CIC filter: introduces a **group delay** $\tau_g = N(R-1)/2 = 10.5$ samples (linear phase), not a phase offset.

### What CAN affect phase
1. **DDS:** introduces a linear phase ramp at $f_{\text{DDS}}$ — does not shift the tone phase after down-conversion to DC.
2. **Acquisition start time:** as in Notebook 02, phases are not reproducible between captures unless RF and ADC clocks share a 10 MHz reference. Large phase spread is expected and normal.

---

## Code: Complete Calibration with DDS+CIC


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, re

# ============================================================
# SYSTEM PARAMETERS  (consistent with Notebooks 01 and 02)
# ============================================================
FS      = 2**15    # Digital full-scale (12-bit ADC MSB-aligned to 16-bit)
fs_pfb  = 256e6    # PFB channel sample rate [Hz]

# ── ADC calibration (Notebook 01) ───────────────────────────
K_ADC = -12.89     # dB: P_adc(dBFS) = P_in(dBm) + K_ADC

# ── PFB (Notebook 02 corrected model) ───────────────────────
PFB_DECIMATION = 8
PFB_QOUT       = 6

G_polyphase = 20 * np.log10(PFB_DECIMATION)     # +18.06 dB
G_qout      = -20 * np.log10(2**PFB_QOUT)       # -36.12 dB
G_PFB_net   = G_polyphase + G_qout              # -18.06 dB

# Calibration constant at PFB output (from Notebook 02)
CAL_CONSTANT_PRE = -K_ADC - G_PFB_net           # +30.95 dB

# ── DDS Mixer ───────────────────────────────────────────────
G_DDS_DB     = 20 * np.log10(2**15)             # +90.31 dB

# ── QPROD quantization (before CIC) ─────────────────────────
QPROD_REG         = 14
G_QPROD_DB        = -20 * np.log10(2**QPROD_REG)   # -84.29 dB

# ── CIC + QCIC (hardware normalized) ────────────────────────
# The CIC has theoretical gain of 512 (+54.19 dB) but the hardware
# design (sign extension to 46 bits + LSB selection with QCIC=0)
# results in a net effective gain of 0 dB.
G_CIC_QCIC_DB = 0.0                              # Effective gain: 0 dB

# ── DDS+CIC block net gain ───────────────────────────────────
G_DDS_CIC_BLOCK = G_DDS_DB + G_QPROD_DB + G_CIC_QCIC_DB   # +6.02 dB

# ── DDS+CIC block net effect on calibration constant ────────
#   Delta = -G_DDS_CIC_BLOCK (relative to PFB output)
DDS_CIC_DELTA_DB  = -G_DDS_CIC_BLOCK              # -6.02 dB

# ── Analytical calibration constant at DDS+CIC output ───────
CAL_CONSTANT_ANALYTICAL = CAL_CONSTANT_PRE + DDS_CIC_DELTA_DB   # +24.93 dB

# Output parameters
R, M, N_cic  = 8, 1, 3
fs_out       = fs_pfb / R                       # 32 MHz
tone_bins = 3

print("=== Signal Chain Parameters ===")
print(f"  FS (digital full-scale):               2^15 = {FS}")
print(f"  K_ADC (Notebook 01):                   {K_ADC:.2f} dB")
print()
print("=== PFB Gain Budget (Notebook 02 corrected model) ===")
print(f"  Polyphase accumulation (D=8):          {G_polyphase:+.2f} dB")
print(f"  qout right-shift (qout=6):             {G_qout:+.2f} dB")
print(f"  Net PFB gain G_PFB_net:                {G_PFB_net:+.2f} dB")
print(f"  CAL_CONSTANT_PRE (PFB output):         {CAL_CONSTANT_PRE:.2f} dB")
print()
print("=== DDS+CIC Block ===")
print(f"  DDS Mixer gain:                        {G_DDS_DB:+.2f} dB")
print(f"  QPROD scaling (QPROD={QPROD_REG}):            {G_QPROD_DB:+.2f} dB")
print(f"  CIC + QCIC (hardware normalized):      {G_CIC_QCIC_DB:+.2f} dB")
print(f"  DDS+CIC block total gain:              {G_DDS_CIC_BLOCK:+.2f} dB")
print(f"  DDS+CIC net effect on CAL_CONSTANT:    {DDS_CIC_DELTA_DB:+.2f} dB")
print()
print("=== Calibration at DDS+CIC Output ===")
print(f"  CAL_CONSTANT_ANALYTICAL:               {CAL_CONSTANT_ANALYTICAL:.2f} dB")
print(f"  Formula: P_in (dBm) = P_out (dBFS) + {CAL_CONSTANT_ANALYTICAL:.2f}")
print()

# ============================================================
# LOAD DATA
# ============================================================
data_path = "data/f_300*/x_postddscic_t_data.txt"
files = sorted(glob.glob(data_path),
               key=lambda f: float(re.search(r'(-?\+?\d+)dBm', f).group(1)))

measurements = []

print(f"{'Pin (dBm)':>10}  {'P_out (dBFS)':>13}  {'Phase (°)':>10}  {'SNR (dB)':>9}  {'Peak (%)':>9}")
print("-" * 60)

# ============================================================
# PROCESS FILES
# ============================================================
for file in files:
    Pin_gen   = float(re.search(r'(-?\+?\d+)dBm', file).group(1))
    x         = np.loadtxt(file, dtype=np.complex128)
    N_samples = len(x)
    max_val   = np.max(np.abs(x))
    std_val   = np.std(np.abs(x))

    # Window & FFT
    window        = np.hanning(N_samples)
    coherent_gain = np.sum(window) / N_samples
    xw            = x * window
    fft_complex   = np.fft.fftshift(np.fft.fft(xw) / N_samples)
    freqs         = np.fft.fftshift(np.fft.fftfreq(N_samples, 1 / fs_out))
    mag           = np.abs(fft_complex) / coherent_gain

    # Peak detection & tone power
    peak_idx       = np.argmax(mag)
    f_peak         = freqs[peak_idx]
    idx0, idx1     = max(0, peak_idx - tone_bins), min(N_samples, peak_idx + tone_bins + 1)
    tone_power     = np.sum(mag[idx0:idx1]**2)
    P_output_dbfs  = 10 * np.log10(tone_power / FS**2)
    tone_phase     = np.angle(fft_complex[peak_idx])

    # SNR & SFDR
    noise_mask             = np.ones(len(mag), dtype=bool)
    noise_mask[idx0:idx1]  = False
    noise_power            = np.mean(mag[noise_mask]**2)
    snr_db                 = 10 * np.log10(tone_power / (len(mag) * noise_power))
    mag_no_tone            = mag.copy()
    mag_no_tone[idx0:idx1] = 0
    sfdr_db                = 10 * np.log10(tone_power / np.max(mag_no_tone)**2)

    measurements.append({
        'Pin_gen':       Pin_gen,
        'P_output_dbfs': P_output_dbfs,
        'f_peak':        f_peak,
        'phase':         tone_phase,
        'max_val':       max_val,
        'std_val':       std_val,
        'snr_db':        snr_db,
        'sfdr_db':       sfdr_db,
        'file':          file,
    })

    saturation_pct = (max_val / FS) * 100
    sat_flag = " ⚠️ SAT" if saturation_pct > 90 else ""
    print(f"{Pin_gen:10.1f}  {P_output_dbfs:13.2f}  "
          f"{np.degrees(tone_phase):10.1f}  {snr_db:9.1f}  {saturation_pct:8.1f}%{sat_flag}")

# ============================================================
# EMPIRICAL CALIBRATION — linear fit on linear-region data
# ============================================================
linear_meas = [m for m in measurements if m['Pin_gen'] <= 0]
Pin_vals    = np.array([m['Pin_gen']       for m in linear_meas])
Pout_vals   = np.array([m['P_output_dbfs'] for m in linear_meas])
coeffs_fit  = np.polyfit(Pout_vals, Pin_vals, 1)
slope              = coeffs_fit[0]
CAL_CONSTANT_POST  = coeffs_fit[1]   # empirical calibration constant

print(f"\n=== Empirical linear fit (linear region, Pin ≤ 0 dBm) ===")
print(f"  Pin = {slope:.4f} × P_out + {CAL_CONSTANT_POST:.2f} dB")
print(f"  Expected slope:      1.0000  (actual: {slope:.4f})")
print(f"  Analytical constant: {CAL_CONSTANT_ANALYTICAL:.2f} dB")
print(f"  Empirical constant:  {CAL_CONSTANT_POST:.2f} dB")
print(f"  Agreement:           {abs(CAL_CONSTANT_POST - CAL_CONSTANT_ANALYTICAL):.2f} dB")

# ============================================================
# GAIN BUDGET VERIFICATION
# ============================================================
print(f"\n{'='*70}")
print("GAIN BUDGET VERIFICATION")
print(f"{'='*70}")

DDS_CIC_NET_MEASURED = CAL_CONSTANT_POST - CAL_CONSTANT_PRE

print(f"  CAL_CONSTANT_PRE  (PFB output, Notebook 02): {CAL_CONSTANT_PRE:.2f} dB")
print(f"  CAL_CONSTANT_POST (DDS+CIC output, measured): {CAL_CONSTANT_POST:.2f} dB")
print(f"  Measured DDS+CIC delta:   {DDS_CIC_NET_MEASURED:.2f} dB")
print(f"  Analytical DDS+CIC delta: {DDS_CIC_DELTA_DB:.2f} dB")
print(f"    (= -G_DDS - G_QPROD - G_CIC_QCIC = -{G_DDS_DB:.2f} - ({G_QPROD_DB:.2f}) - {G_CIC_QCIC_DB:.2f} = {DDS_CIC_DELTA_DB:.2f} dB)")
print(f"  Model vs measurement error: {abs(DDS_CIC_NET_MEASURED - DDS_CIC_DELTA_DB):.2f} dB")

# ============================================================
# PHASE ANALYSIS
# ============================================================
Pin_list   = [m['Pin_gen']           for m in measurements]
phases_deg = [np.degrees(m['phase']) for m in measurements]
phase_mean = np.mean(phases_deg)
phase_std  = np.std(phases_deg)
phase_power_corr = np.corrcoef(Pin_list, phases_deg)[0, 1]

print(f"\n=== Phase Analysis ===")
print(f"  Mean:  {phase_mean:.2f}°")
print(f"  Std:   {phase_std:.2f}°  (large spread is expected — no phase lock between RF gen and ADC clock)")
print(f"  Phase–Power correlation: {phase_power_corr:.3f}")

# ============================================================
# POWER RECOVERY — apply empirical CAL_CONSTANT_POST
# ============================================================
Pin_list      = [m['Pin_gen']       for m in measurements]
Pin_recovered = [m['P_output_dbfs'] + CAL_CONSTANT_POST for m in measurements]
errors        = np.array(Pin_recovered) - np.array(Pin_list)

lin_errors = [e for e, p in zip(errors, Pin_list) if p <= 0]
print(f"\n=== Recovery Accuracy (linear region, Pin ≤ 0 dBm) ===")
print(f"  Mean error:  {np.mean(lin_errors):+.3f} dB")
print(f"  Std dev:      {np.std(lin_errors):.3f} dB")
print(f"  Max |error|:  {np.max(np.abs(lin_errors)):.3f} dB")
print(f"  RMS error:    {np.sqrt(np.mean(np.array(lin_errors)**2)):.3f} dB")

# ============================================================
# PLOTTING
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs  = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.35)
snr_list  = [m['snr_db']  for m in measurements]
sfdr_list = [m['sfdr_db'] for m in measurements]
max_vals  = [m['max_val'] for m in measurements]

# ── Calibration ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(Pin_list, Pin_recovered, 'o-', ms=8, lw=2, color='steelblue', label='Recovered')
ax1.plot(Pin_list, Pin_list, 'k--', lw=2, label='Ideal (1:1)')
ax1.set_xlabel('True Input Power (dBm)', fontsize=11)
ax1.set_ylabel('Recovered Power (dBm)', fontsize=11)
ax1.set_title('Post-DDS+CIC Calibration', fontsize=12, fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# ── Error ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(Pin_list, errors, 'o-', ms=8, lw=2, color='seagreen')
ax2.axhline(0, color='k', ls='--', lw=2)
ax2.axhline( 0.5, color='orange', ls=':', lw=1.5, alpha=0.8, label='±0.5 dB')
ax2.axhline(-0.5, color='orange', ls=':', lw=1.5, alpha=0.8)
ax2.fill_between(Pin_list, -0.5, 0.5, alpha=0.12, color='seagreen')
ax2.set_xlabel('Input Power (dBm)', fontsize=11)
ax2.set_ylabel('Error (dB)', fontsize=11)
ax2.set_title(f'Calibration Error  (σ={np.std(lin_errors):.3f} dB, linear region)', fontsize=12, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

# ── Phase vs Power ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
scatter = ax3.scatter(Pin_list, phases_deg, c=snr_list, s=100,
                      cmap='viridis', edgecolors='k', linewidth=1)
ax3.axhline(phase_mean, color='r', ls='--', lw=2, alpha=0.7, label=f'μ={phase_mean:.1f}°')
if abs(phase_power_corr) > 0.5:
    coeffs_phase = np.polyfit(Pin_list, phases_deg, 1)
    p_fit = np.poly1d(coeffs_phase)
    ax3.plot(Pin_list, p_fit(Pin_list), 'r-', lw=2, alpha=0.5,
             label=f'r={phase_power_corr:.2f}')
ax3.legend()
ax3.set_xlabel('Input Power (dBm)', fontsize=11)
ax3.set_ylabel('Phase (degrees)', fontsize=11)
ax3.set_title(f'Phase vs Power (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='SNR (dB)')
ax3.grid(True, alpha=0.3)

# ── Saturation Check ──────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(Pin_list, max_vals, 'o-', ms=8, lw=2, color='red')
ax4.axhline(FS, color='k', ls='-', lw=2, label='Full Scale')
ax4.axhline(0.9 * FS, color='orange', ls='--', lw=2, label='90% FS')
ax4.set_xlabel('Input Power (dBm)', fontsize=11)
ax4.set_ylabel('Peak Amplitude', fontsize=11)
ax4.set_title('Saturation Check', fontsize=12, fontweight='bold')
ax4.legend(); ax4.grid(True, alpha=0.3)

# ── SNR ───────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(Pin_list, snr_list, 'o-', ms=8, lw=2, color='green')
ax5.set_xlabel('Input Power (dBm)', fontsize=11)
ax5.set_ylabel('SNR (dB)', fontsize=11)
ax5.set_title('Signal-to-Noise Ratio', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# ── SFDR ──────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(Pin_list, sfdr_list, 'o-', ms=8, lw=2, color='purple')
ax6.set_xlabel('Input Power (dBm)', fontsize=11)
ax6.set_ylabel('SFDR (dB)', fontsize=11)
ax6.set_title('Spurious-Free Dynamic Range', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

# ── Phase Histogram ───────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
ax7.hist(phases_deg, bins=15, edgecolor='k', alpha=0.7, color='purple')
ax7.axvline(phase_mean, color='r', ls='--', lw=2, label=f'μ={phase_mean:.1f}°')
ax7.axvline(phase_mean - phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.axvline(phase_mean + phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.set_xlabel('Phase (degrees)', fontsize=11)
ax7.set_ylabel('Count', fontsize=11)
ax7.set_title(f'Phase Distribution (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
ax7.legend(); ax7.grid(True, alpha=0.3, axis='y')

# ── Frequency Stability ───────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 1])
freqs_peak = [m['f_peak'] / 1e6 for m in measurements]
ax8.plot(Pin_list, freqs_peak, 'o-', ms=8, lw=2, color='orange')
ax8.axhline(np.mean(freqs_peak), color='r', ls='--', lw=2,
            label=f'μ={np.mean(freqs_peak):.4f} MHz')
ax8.set_xlabel('Input Power (dBm)', fontsize=11)
ax8.set_ylabel('Peak Frequency (MHz)', fontsize=11)
ax8.set_title('Frequency After DDS (residual offset)', fontsize=12, fontweight='bold')
ax8.legend(); ax8.grid(True, alpha=0.3)

# ── IQ Constellation ──────────────────────────────────────────
ax9 = fig.add_subplot(gs[2, 2])
colors_cm = plt.cm.plasma(np.linspace(0, 1, len(measurements)))
for i, (m, c) in enumerate(zip(measurements[::2], colors_cm[::2])):
    x_data    = np.loadtxt(m['file'], dtype=np.complex128)
    subsample = max(1, len(x_data) // 400)
    label     = f"{m['Pin_gen']:+.0f} dBm" if i % 2 == 0 else ""
    ax9.scatter(np.real(x_data[::subsample]), np.imag(x_data[::subsample]),
                alpha=0.3, s=1, color=c, label=label)
ax9.set_xlabel('I (Real)', fontsize=11)
ax9.set_ylabel('Q (Imaginary)', fontsize=11)
ax9.set_title('IQ Constellation', fontsize=12, fontweight='bold')
ax9.legend(fontsize=8, markerscale=5, ncol=2)
ax9.grid(True, alpha=0.3); ax9.axis('equal')

# ── Calibrated Spectrum (mid-power) ───────────────────────────
ax10 = fig.add_subplot(gs[3, :2])
mid_idx    = len(measurements) // 2
x_mid      = np.loadtxt(measurements[mid_idx]['file'], dtype=np.complex128)
window_mid = np.hanning(len(x_mid))
X_mid      = np.fft.fftshift(np.fft.fft(x_mid * window_mid) / len(x_mid))
f_mid      = np.fft.fftshift(np.fft.fftfreq(len(x_mid), 1 / fs_out))
# Amplitude spectrum normalised to FS, then shifted to dBm via empirical CAL_CONSTANT_POST
coherent_gain_mid = np.sum(window_mid) / len(x_mid)
mag_mid_dBm = 20 * np.log10(np.abs(X_mid) / coherent_gain_mid / FS) + CAL_CONSTANT_POST
ax10.plot(f_mid / 1e6, mag_mid_dBm, lw=1, alpha=0.7)
ax10.axvline(0, color='r', ls='--', lw=2, alpha=0.5, label='DC (after DDS)')
ax10.set_xlabel('Frequency (MHz)', fontsize=11)
ax10.set_ylabel('Power (dBm)', fontsize=11)
ax10.set_title(f'Calibrated Spectrum  (Pin={Pin_list[mid_idx]:+.0f} dBm)',
               fontsize=12, fontweight='bold')
ax10.legend(); ax10.grid(True, alpha=0.3)
ax10.set_xlim([-fs_out / 2 / 1e6, fs_out / 2 / 1e6])

# ── Summary box ───────────────────────────────────────────────
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis('off')
summary_text = (
    f"CALIBRATION SUMMARY\n\n"
    f"K_ADC:            {K_ADC:.2f} dB\n"
    f"G_polyphase(D=8): +18.06 dB\n"
    f"G_qout(qout=6):   -36.12 dB\n"
    f"G_PFB_net:        {G_PFB_net:.2f} dB\n"
    f"G_DDS_MIXER:      +{G_DDS_DB:.2f} dB\n"
    f"G_QPROD (14):     {G_QPROD_DB:.2f} dB\n"
    f"G_CIC+QCIC:       {G_CIC_QCIC_DB:.2f} dB\n\n"
    f"CAL_PRE  (Nb 02): {CAL_CONSTANT_PRE:.2f} dB\n"
    f"CAL_POST (analyt):{CAL_CONSTANT_ANALYTICAL:.2f} dB\n"
    f"CAL_POST (empir): {CAL_CONSTANT_POST:.2f} dB\n"
    f"Agreement:        {abs(CAL_CONSTANT_POST-CAL_CONSTANT_ANALYTICAL):.2f} dB\n\n"
    f"Accuracy:        ±{np.std(lin_errors):.3f} dB\n"
    f"Phase σ:          {phase_std:.2f}°\n"
    f"SNR range:        {np.min(snr_list):.1f}–{np.max(snr_list):.1f} dB"
)
ax11.text(0.05, 0.5, summary_text, transform=ax11.transAxes,
          fontsize=10, verticalalignment='center', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig('images/ddscic_complete_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# FINAL REPORT
# ============================================================
print(f"\n{'='*70}")
print("FINAL DIAGNOSTIC REPORT")
print(f"{'='*70}")

print(f"\n GAIN BUDGET:")
print(f"  K_ADC:                     {K_ADC:.2f} dB  (Notebook 01)")
print(f"  G_polyphase (D=8):         {G_polyphase:+.2f} dB  (Notebook 02)")
print(f"  G_qout (qout=6):           {G_qout:+.2f} dB  (Notebook 02)")
print(f"  G_PFB_net:                 {G_PFB_net:+.2f} dB")
print(f"  G_DDS (Mixer 16-bit):      +{G_DDS_DB:.2f} dB")
print(f"  G_QPROD (QPROD=14):        {G_QPROD_DB:+.2f} dB")
print(f"  G_CIC+QCIC (normalized):   {G_CIC_QCIC_DB:+.2f} dB")

print(f"\n CALIBRATION:")
print(f"  CAL_CONSTANT_PRE  (Nb 02): {CAL_CONSTANT_PRE:.2f} dB")
print(f"  CAL_CONSTANT_POST (analyt):{CAL_CONSTANT_ANALYTICAL:.2f} dB")
print(f"  CAL_CONSTANT_POST (empir): {CAL_CONSTANT_POST:.2f} dB")
print(f"  Agreement:                 {abs(CAL_CONSTANT_POST-CAL_CONSTANT_ANALYTICAL):.2f} dB")
print(f"  Formula: P_in (dBm) = P_out (dBFS) + {CAL_CONSTANT_POST:.2f}")
print(f"  Linear fit accuracy:      ±{np.std(lin_errors):.3f} dB (linear region)")

print(f"\n PHASE:")
if phase_std < 10:
    print(f"   Excellent phase stability: {phase_std:.2f}° std dev")
elif phase_std < 30:
    print(f"   Good: {phase_std:.2f}° std dev (some variation expected without phase lock)")
elif abs(phase_power_corr) > 0.7:
    print(f"  ! Phase depends on power (r={phase_power_corr:.2f}) — check non-linearity")
else:
    print(f"  ! Large variation ({phase_std:.2f}°) — expected if RF gen and ADC clock are not phase-locked")

print(f"\n NOTES:")
print(f"  • The CIC is ACTIVE (decimation R=8, M=1, N=3)")
print(f"  • QCIC_REG=0 selects bits [15:0] of the 46-bit CIC output")
print(f"  • The combination of sign extension + CIC gain + LSB selection results in G_CIC+QCIC = 0 dB")
print(f"  • This is an intentional hardware normalization")
